Reading the tables data

## Reading data into dataframe

In [0]:
from pyspark.sql.functions import col, sum, when ,count,countDistinct

In [0]:
date='2026-10-26'

In [0]:

organizations_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://health-care-project@stheathproject.dfs.core.windows.net/bronze/organizations/{date}/organizations.csv") 
organizations_df.printSchema()


## Fixed the coulumn names and type

In [0]:
# Fixed the coulumn names


organizations_df=organizations_df.withColumnRenamed('Id','ORGANIZATION_id').withColumnRenamed('Name','ORGANIZATION_name')
organizations_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in organizations_df.columns
]).show(truncate=False)



In [0]:
from pyspark.sql import functions as F

# 1. Select the Bronze Encounters DataFrame
df = organizations_df

# 2. Standardize column names
for col_name in df.columns:
    new_name = (
        col_name.strip()
        .lower()
        .replace(" ", "_")
    )
    df = df.withColumnRenamed(col_name, new_name)

# 3. Inspect the transformed schema
df.printSchema()

# 4. Validate row count
print("Silver input rows:", df.count())

# 5. Check primary-key nulls
display(
    df.filter(F.col("organization_id").isNull())
)

# 6. Check duplicate encounter IDs
display(
    df.groupBy("organization_id")
      .count()
      .filter(F.col("count") > 1)
)


In [0]:
silver_path = f"abfss://health-care-project@stheathproject.dfs.core.windows.net/silver/Organization/{date}"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
silver_df=spark.read\
    .format('delta')\
    .load(silver_path)



print("Bronze/input rows:", df.count())
print("Silver rows:", silver_df.count())

display(silver_df.limit(10))

In [0]:
path = f"abfss://health-care-project@stheathproject.dfs.core.windows.net/silver/Organization/{date}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS health_care.silver.silver_organization
USING DELTA
LOCATION '{path}'
""")